# Hybrid residual GRAPE

Closed-loop Fock-state optimization with a fixed physics model plus a simple RBF/ridge residual correction.

The inner optimizer uses a differentiable surrogate model

```text
logit(P_pred(u)) = logit(P_phys(u, p1)) + support(u) * f_RBF(u, p2)
```

where `u` is the 80-dimensional B-spline control vector, `p1` is the fixed Hamiltonian model, and `p2` is fitted from binary measurement data. `support(u)` is close to 1 near previously measured pulses and close to 0 far away, so the residual correction is local instead of being trusted everywhere.

This is intentionally closer to GRAPE than SPSA/CEM: the pulse optimizer uses `jax.grad` through the physics simulator plus the learned correction. The experiment still only provides binomial measurements.


In [ ]:
from dataclasses import replace
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from hybrid_residual_grape import (
    FockPhysicsModel,
    HybridGrapeConfig,
    PhysicsParams,
    RBFResidualConfig,
    SimulationConfig,
    append_dataset,
    empty_rbf_model,
    fit_rbf_residual,
    make_local_experiment_batch,
    optimize_hybrid_grape,
    sample_binomial_measurements,
)
from hybrid_residual_grape.config import khz_to_rad_per_us
from hybrid_residual_grape.residual import (
    hybrid_probability_from_physics,
    rbf_diagnostics,
    residual_logit,
    rbf_support,
)

jax.config.update("jax_enable_x64", True)


## Setup

The `physics_model` is what the optimizer believes. The `true_model` is only for this notebook's simulated experiment. In the real OPX1000/DGX setup, replace `measure_on_experiment` with the hardware call and delete/ignore the hidden true diagnostics.


In [ ]:
seed = 1234
key = jax.random.key(seed)

q = SimulationConfig(
    n_cav=25,
    target_n=2,
    initial_cavity_n=0,
    initial_qubit_state=0,
    t_drive=1.408,
    ndt_drive=80,
    num_coeffs=20,
    spline_degree=2,
    spline_skip_left=2,
    spline_skip_right=2,
    param_clip=2.0,
)

p1 = PhysicsParams()
physics_model = FockPhysicsModel(q, p1)

# Hidden small mismatch used only to simulate the real experiment in this notebook.
true_params = PhysicsParams(
    chi=p1.chi + khz_to_rad_per_us(3.0),
    cavity_self_kerr=p1.cavity_self_kerr + khz_to_rad_per_us(0.12),
    cavity_detuning=khz_to_rad_per_us(2.0),
    qubit_detuning=khz_to_rad_per_us(-3.0),
    mu_qub=p1.mu_qub * 1.010,
    mu_cav=p1.mu_cav * 0.988,
    cavity_phase=0.025,
)
true_model = FockPhysicsModel(q, true_params)

print("parameter size:", physics_model.parameter_size)
print("basis endpoint max:", float(jnp.max(jnp.abs(physics_model.bsplines_edges[:, [0, -1]]))))
print("max active B-splines:", int(jnp.max(jnp.sum(physics_model.bsplines_mids > 1e-12, axis=0))))
print("chi [rad/us]:", p1.chi)
print("self Kerr [rad/us]:", p1.cavity_self_kerr)


## Real experiment hook

For now this samples a binomial distribution from the hidden simulator. On hardware, this function should send each `controls[i]` to the OPX1000, run `shots` repetitions of the selective photon-number test, and return successes and shot counts.

One shot is about `400 us` with reset, so `batch_size * shots * 400 us` is the measurement time per round.


In [ ]:
shot_time_us = 400.0


def measure_on_experiment(controls, key, shots):
    return sample_binomial_measurements(
        true_model,
        controls,
        key,
        shots=shots,
    )


## Hyperparameters

These settings are chosen for a fairer comparison than a quick smoke test:

- the inner GRAPE/L-BFGS solve runs longer, because under-converged GRAPE makes the baseline artificially weak;
- the closed-loop experiment runs twice as many rounds as before;
- the RBF residual is regularized and support-gated, so it can correct measured model error near data without inventing a global Hamiltonian correction from sparse shots.

For real OPX/DGX runs, tune `experiment_batch_size * shots_per_pulse * 400 us` so the measurement time per round is comparable to the DGX training time per round.


In [ ]:
rbf_config = RBFResidualConfig(
    max_centers=192,
    length_scale=0.20,
    ridge=1e-2,
    residual_clip=0.8,
    measurement_floor=1e-3,
)

grape_config = HybridGrapeConfig(
    maxiter=220,
    memory_size=20,
    noise_samples=5,
    control_noise_std=0.015,
    residual_support_penalty=0.08,
    residual_size_penalty=0.03,
    amplitude_l2=5e-5,
    smoothness_l2=5e-5,
    param_clip=q.param_clip,
    grad_clip_norm=30.0,
)

# Keep the number of starts modest; make each solve more converged instead.
physics_only_restarts = 6
physics_only_maxiter = 320

rounds = 24
experiment_batch_size = 12
shots_per_pulse = 1000
local_noise_std = 0.025

# OPX/DGX rule of thumb: increase experiment_batch_size and shots_per_pulse
# until measurement time is similar to the time spent fitting + GRAPE on DGX.
round_measurement_time_s = experiment_batch_size * shots_per_pulse * shot_time_us * 1e-6
print("GRAPE maxiter per hybrid round:", grape_config.maxiter)
print("physics-only baseline maxiter:", physics_only_maxiter)
print("closed-loop rounds:", rounds)
print("binary measurements per round:", experiment_batch_size * shots_per_pulse)
print("measurement time per round [s]:", round_measurement_time_s)
print("total planned binary measurements:", rounds * experiment_batch_size * shots_per_pulse)


## Initialize state

We start from a small random pulse instead of exactly zero. For target Fock states, the exact zero pulse can have an unhelpful flat gradient.


In [ ]:
key, init_key = jax.random.split(key)
best_controls = 0.02 * jax.random.normal(init_key, (physics_model.parameter_size,))
best_controls = jnp.clip(best_controls, -q.param_clip, q.param_clip)

residual_model = empty_rbf_model(physics_model.parameter_size, rbf_config)

dataset_controls = None
dataset_successes = None
dataset_shots = None
dataset_physics_probability = None

progress_rows = []
grape_histories = []

total_measurements = 0
best_measured = -1.0
best_true_diagnostic = float(true_model.photon_probability(best_controls))
print("initial true diagnostic P_n:", best_true_diagnostic)


## Physics-only GRAPE baseline

This baseline runs a full GRAPE optimization on the nominal physics model only, with no RBF correction and no experiment feedback. After the nominal-model pulse is optimized, we evaluate that same pulse on the slightly modified hidden Hamiltonian to see how much model mismatch hurts.

The important fairness point is that this cell should be allowed to converge. We therefore use longer L-BFGS runs, but still only a modest number of restarts.


In [ ]:
physics_only_residual = empty_rbf_model(physics_model.parameter_size, rbf_config)
physics_only_config = replace(
    grape_config,
    maxiter=physics_only_maxiter,
    noise_samples=1,
    residual_support_penalty=0.0,
    residual_size_penalty=0.0,
    control_noise_std=0.0,
)

# Baseline definition:
# 1. run a full GRAPE optimization on the nominal physics model only;
# 2. select the pulse with the best nominal-model fidelity;
# 3. only afterwards evaluate that pulse on the hidden true Hamiltonian.
# Multiple restarts avoid reporting an accidentally bad local optimum.
key, restart_key = jax.random.split(key)
restart_controls = 0.04 * jax.random.normal(
    restart_key,
    (physics_only_restarts, physics_model.parameter_size),
)
restart_controls = restart_controls.at[0].set(best_controls)
restart_controls = jnp.clip(restart_controls, -q.param_clip, q.param_clip)

physics_only_candidates = []
physics_only_histories = []
key, baseline_key = jax.random.split(key)
baseline_keys = jax.random.split(baseline_key, physics_only_restarts)

for restart_index in tqdm(range(physics_only_restarts), desc="physics-only GRAPE restarts"):
    candidate, history, summary, _ = optimize_hybrid_grape(
        physics_model,
        physics_only_residual,
        restart_controls[restart_index],
        baseline_keys[restart_index],
        physics_only_config,
    )
    physics_only_candidates.append(candidate)
    physics_only_histories.append(history)

physics_only_candidates = jnp.stack(physics_only_candidates)
physics_only_pred_all = physics_model.population_probability(physics_only_candidates)
physics_only_true_all = true_model.population_probability(physics_only_candidates)
physics_only_best_idx = int(jnp.argmax(physics_only_pred_all))
physics_only_controls = physics_only_candidates[physics_only_best_idx]
physics_only_pred = physics_only_pred_all[physics_only_best_idx]
physics_only_true = physics_only_true_all[physics_only_best_idx]
physics_only_history = physics_only_histories[physics_only_best_idx]

print("physics-only GRAPE restarts:", physics_only_restarts)
print("physics-only maxiter per restart:", physics_only_config.maxiter)
print("physics-only best restart:", physics_only_best_idx)
print("physics-only nominal-model P_n:", float(physics_only_pred))
print("physics-only true-Hamiltonian diagnostic P_n:", float(physics_only_true))
print("physics-only true-Hamiltonian -log10(1-P_n):", float(-jnp.log10(jnp.maximum(1.0 - physics_only_true, 1e-8))))
print("all restart nominal-model P_n:", jax.device_get(physics_only_pred_all))
print("all restart true-Hamiltonian P_n:", jax.device_get(physics_only_true_all))


## Run closed-loop hybrid residual GRAPE

Each round fits the RBF residual from all accumulated data, warm-starts GRAPE from the previous best pulse, measures the proposed pulse plus local noisy variants, and appends those measurements to the dataset.


In [ ]:
pbar = tqdm(range(rounds), desc="hybrid residual GRAPE")

for round_index in pbar:
    if dataset_controls is not None and dataset_controls.shape[0] > 0:
        residual_model = fit_rbf_residual(
            dataset_controls,
            dataset_physics_probability,
            dataset_successes,
            dataset_shots,
            rbf_config,
        )
        diag = rbf_diagnostics(
            residual_model,
            dataset_controls,
            dataset_physics_probability,
            dataset_successes,
            dataset_shots,
        )
        fit_wmse = float(diag["weighted_mse"])
        active_centers = float(diag["active_centers"])
    else:
        residual_model = empty_rbf_model(physics_model.parameter_size, rbf_config)
        fit_wmse = float("nan")
        active_centers = 0.0

    proposed_controls, grape_history, grape_summary, key = optimize_hybrid_grape(
        physics_model,
        residual_model,
        best_controls,
        key,
        grape_config,
    )
    grape_histories.append(grape_history)

    experiment_controls, key = make_local_experiment_batch(
        proposed_controls,
        key,
        batch_size=experiment_batch_size,
        noise_std=local_noise_std,
        param_clip=q.param_clip,
        include_center=True,
    )

    physics_probability = physics_model.population_probability(experiment_controls)
    successes, shot_counts, true_probability, key = measure_on_experiment(
        experiment_controls,
        key,
        shots_per_pulse,
    )

    dataset_controls, dataset_successes, dataset_shots, dataset_physics_probability = append_dataset(
        dataset_controls,
        dataset_successes,
        dataset_shots,
        dataset_physics_probability,
        experiment_controls,
        successes,
        shot_counts,
        physics_probability,
    )
    total_measurements += int(experiment_batch_size * shots_per_pulse)

    measured = successes / shot_counts
    best_in_batch = int(jnp.argmax(measured))
    if float(measured[best_in_batch]) > best_measured:
        best_measured = float(measured[best_in_batch])
        best_true_diagnostic = float(true_probability[best_in_batch])
        best_controls = experiment_controls[best_in_batch]
    else:
        # Still warm-start the next inner GRAPE from the proposed center if the
        # high-shot estimate is close. This avoids freezing due to binomial noise.
        if float(measured[0]) > best_measured - 0.03:
            best_controls = proposed_controls

    progress_rows.append(
        jnp.array(
            [
                float(total_measurements),
                best_measured,
                best_true_diagnostic,
                float(measured[0]),
                float(true_probability[0]),
                float(grape_summary[1]),
                float(grape_summary[2]),
                float(grape_summary[3]),
                float(grape_summary[4]),
                fit_wmse,
                active_centers,
                float(dataset_controls.shape[0]),
            ]
        )
    )

    pbar.set_postfix(
        {
            "best_meas": f"{best_measured:.3f}",
            "best_true": f"{best_true_diagnostic:.4f}",
            "center_meas": f"{float(measured[0]):.3f}",
            "pred": f"{float(grape_summary[1]):.3f}",
            "data": dataset_controls.shape[0],
        }
    )

progress = jnp.stack(progress_rows) if progress_rows else jnp.zeros((0, 12))
print("total binary measurements:", total_measurements)
print("best measured P_n:", best_measured)
print("best true diagnostic P_n:", best_true_diagnostic)


## Plot progress

The model proposes pulses, but the pulse we trust is still selected by measured successes/shots. The hidden true curve is only for simulation diagnostics.

Use the second panel, `-log10(1 - P_n)`, for high-fidelity comparisons. Around 99%, ordinary `P_n` plots compress the important differences.


In [ ]:
if len(progress_rows):
    progress = jnp.stack(progress_rows)

    def log_infidelity(probability):
        return -jnp.log10(jnp.maximum(1.0 - probability, 1e-8))

    fig, axes = plt.subplots(4, 1, figsize=(8.5, 10), sharex=True)

    axes[0].plot(progress[:, 0], progress[:, 1], label="best measured")
    axes[0].plot(progress[:, 0], progress[:, 2], label="best true diagnostic")
    axes[0].plot(progress[:, 0], progress[:, 4], label="center true diagnostic", alpha=0.65)
    axes[0].axhline(float(physics_only_true), color="black", linestyle="--", linewidth=1, label="physics-only GRAPE true")
    axes[0].set_ylabel("P_n")
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()

    axes[1].plot(progress[:, 0], log_infidelity(progress[:, 2]), label="best true diagnostic")
    axes[1].plot(progress[:, 0], log_infidelity(progress[:, 4]), label="center true diagnostic")
    axes[1].plot(progress[:, 0], log_infidelity(progress[:, 5]), label="model-predicted center")
    axes[1].axhline(float(log_infidelity(physics_only_true)), color="black", linestyle="--", linewidth=1, label="physics-only GRAPE true")
    axes[1].set_ylabel("-log10(1 - P_n)")
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()

    axes[2].plot(progress[:, 0], progress[:, 5], label="hybrid predicted center")
    axes[2].plot(progress[:, 0], progress[:, 6], label="physics-only center")
    axes[2].plot(progress[:, 0], progress[:, 7], label="RBF support")
    axes[2].plot(progress[:, 0], progress[:, 8], label="|RBF logit correction|")
    axes[2].set_ylabel("model stats")
    axes[2].grid(True, alpha=0.3)
    axes[2].legend()

    ax_mse = axes[3]
    ax_centers = ax_mse.twinx()
    ax_mse.plot(progress[:, 0], jnp.maximum(progress[:, 9], 1e-8), label="RBF weighted MSE", color="tab:blue")
    ax_mse.set_yscale("log")
    ax_mse.set_ylabel("weighted MSE", color="tab:blue")
    ax_mse.tick_params(axis="y", labelcolor="tab:blue")
    ax_centers.plot(progress[:, 0], progress[:, 10], label="active centers", color="tab:orange")
    ax_centers.set_ylabel("active centers", color="tab:orange")
    ax_centers.tick_params(axis="y", labelcolor="tab:orange")
    ax_mse.set_xlabel("total binary measurements")
    ax_mse.grid(True, alpha=0.3)
    plt.show()
else:
    print("Run the loop first.")


## Did the RBF correction learn something significant?

The RBF is useful only if it corrects a real, repeatable discrepancy between the nominal physics model and the measured experiment.

In this simulated notebook we can check this directly because the hidden true Hamiltonian is known. On hardware, the same logic is applied with `measured successes/shots` replacing the hidden true probability:

- `hybrid - physics` should have the same sign and approximate size as `true - physics` or `measured - physics`;
- the absolute error after the RBF should be smaller than the absolute error before the RBF;
- large corrections should occur where RBF support is high, otherwise the model is extrapolating.

If all correction plots sit near zero and the before/after errors are identical, the RBF is not doing anything meaningful for that mismatch.


In [ ]:
if dataset_controls is not None and dataset_controls.shape[0] > 0:
    residual_model = fit_rbf_residual(
        dataset_controls,
        dataset_physics_probability,
        dataset_successes,
        dataset_shots,
        rbf_config,
    )
    diag = rbf_diagnostics(
        residual_model,
        dataset_controls,
        dataset_physics_probability,
        dataset_successes,
        dataset_shots,
    )
    true_diag = true_model.population_probability(dataset_controls)

    measured = diag["measured"]
    physics_p = dataset_physics_probability
    hybrid_p = diag["predicted"]
    learned_probability_correction = hybrid_p - physics_p
    measured_discrepancy = measured - physics_p
    hidden_discrepancy = true_diag - physics_p
    physics_error_to_measured = measured - physics_p
    hybrid_error_to_measured = measured - hybrid_p
    physics_error_to_true = true_diag - physics_p
    hybrid_error_to_true = true_diag - hybrid_p

    def log_infidelity(probability):
        return -jnp.log10(jnp.maximum(1.0 - probability, 1e-8))

    correction_limit = float(
        jnp.maximum(
            0.03,
            1.05
            * jnp.max(
                jnp.abs(
                    jnp.concatenate(
                        [hidden_discrepancy, learned_probability_correction, measured_discrepancy]
                    )
                )
            ),
        )
    )

    fig, axes = plt.subplots(2, 3, figsize=(15, 8.5))
    axes = axes.ravel()

    sc0 = axes[0].scatter(physics_p, measured, c=diag["support"], cmap="viridis", alpha=0.8)
    axes[0].plot([0, 1], [0, 1], color="black", linewidth=1)
    axes[0].set_xlabel("physics-only P_n")
    axes[0].set_ylabel("measured successes/shots")
    axes[0].set_title("before correction")
    axes[0].grid(True, alpha=0.3)
    fig.colorbar(sc0, ax=axes[0], label="RBF support")

    sc1 = axes[1].scatter(hybrid_p, measured, c=diag["residual"], cmap="coolwarm", alpha=0.8)
    axes[1].plot([0, 1], [0, 1], color="black", linewidth=1)
    axes[1].set_xlabel("hybrid predicted P_n")
    axes[1].set_ylabel("measured successes/shots")
    axes[1].set_title("after correction")
    axes[1].grid(True, alpha=0.3)
    fig.colorbar(sc1, ax=axes[1], label="support-gated logit correction")

    sc2 = axes[2].scatter(
        hidden_discrepancy,
        learned_probability_correction,
        c=diag["support"],
        cmap="viridis",
        alpha=0.85,
    )
    axes[2].plot(
        [-correction_limit, correction_limit],
        [-correction_limit, correction_limit],
        color="black",
        linewidth=1,
    )
    axes[2].axhline(0.0, color="gray", linewidth=0.8)
    axes[2].axvline(0.0, color="gray", linewidth=0.8)
    axes[2].set_xlim(-correction_limit, correction_limit)
    axes[2].set_ylim(-correction_limit, correction_limit)
    axes[2].set_xlabel("hidden true - physics")
    axes[2].set_ylabel("hybrid - physics")
    axes[2].set_title("did RBF learn the true mismatch?")
    axes[2].grid(True, alpha=0.3)
    fig.colorbar(sc2, ax=axes[2], label="RBF support")

    bins = jnp.linspace(
        0.0,
        float(jnp.maximum(jnp.max(jnp.abs(physics_error_to_measured)), jnp.max(jnp.abs(hybrid_error_to_measured)))) + 1e-6,
        35,
    )
    axes[3].hist(jax.device_get(jnp.abs(physics_error_to_measured)), bins=jax.device_get(bins), alpha=0.55, label="|measured - physics|")
    axes[3].hist(jax.device_get(jnp.abs(hybrid_error_to_measured)), bins=jax.device_get(bins), alpha=0.55, label="|measured - hybrid|")
    axes[3].set_xlabel("absolute probability error")
    axes[3].set_ylabel("count")
    axes[3].set_title("measured error before/after RBF")
    axes[3].grid(True, alpha=0.3)
    axes[3].legend()

    sc4 = axes[4].scatter(physics_p, learned_probability_correction, c=diag["residual"], cmap="coolwarm", alpha=0.8)
    axes[4].axhline(0.0, color="black", linewidth=1)
    axes[4].set_xlabel("physics-only P_n")
    axes[4].set_ylabel("hybrid - physics")
    axes[4].set_title("probability correction vs fidelity")
    axes[4].grid(True, alpha=0.3)
    fig.colorbar(sc4, ax=axes[4], label="support-gated logit correction")

    sc5 = axes[5].scatter(diag["support"], jnp.abs(diag["residual"]), c=physics_p, cmap="plasma", alpha=0.85)
    axes[5].set_xlabel("RBF support")
    axes[5].set_ylabel("|logit correction|")
    axes[5].set_title("correction is local if support is high")
    axes[5].grid(True, alpha=0.3)
    fig.colorbar(sc5, ax=axes[5], label="physics-only P_n")

    plt.tight_layout()
    plt.show()

    print("weighted MSE:", float(diag["weighted_mse"]))
    print("active centers:", float(diag["active_centers"]))
    print("mean |RBF logit correction|:", float(jnp.mean(jnp.abs(diag["residual"]))))
    print("max |RBF logit correction|:", float(jnp.max(jnp.abs(diag["residual"]))))
    print("mean |hybrid - physics|:", float(jnp.mean(jnp.abs(learned_probability_correction))))
    print("measured MAE physics -> data:", float(jnp.mean(jnp.abs(physics_error_to_measured))))
    print("measured MAE hybrid  -> data:", float(jnp.mean(jnp.abs(hybrid_error_to_measured))))
    print("hidden true MAE physics -> true:", float(jnp.mean(jnp.abs(physics_error_to_true))))
    print("hidden true MAE hybrid  -> true:", float(jnp.mean(jnp.abs(hybrid_error_to_true))))
else:
    print("No dataset yet.")


## RBF correction along a pulse path

This one-dimensional slice compares the nominal physics model, the hybrid model, and the hidden true experiment along the line from the physics-only GRAPE pulse to the best closed-loop pulse.

The bottom panels are the key: they show whether the RBF correction is supported by nearby measured data and whether the learned probability correction has the right sign compared with the hidden true mismatch.


In [ ]:
if dataset_controls is not None and dataset_controls.shape[0] > 0:
    residual_model = fit_rbf_residual(
        dataset_controls,
        dataset_physics_probability,
        dataset_successes,
        dataset_shots,
        rbf_config,
    )
    t = jnp.linspace(0.0, 1.0, 101)
    line_controls = (1.0 - t[:, None]) * physics_only_controls[None, :] + t[:, None] * best_controls[None, :]
    line_phys = physics_model.population_probability(line_controls)
    line_hybrid = hybrid_probability_from_physics(line_phys, residual_model, line_controls)
    line_true = true_model.population_probability(line_controls)
    line_support = rbf_support(residual_model, line_controls)
    line_residual = residual_logit(residual_model, line_controls)
    line_learned_probability_correction = line_hybrid - line_phys
    line_hidden_discrepancy = line_true - line_phys

    def log_infidelity(probability):
        return -jnp.log10(jnp.maximum(1.0 - probability, 1e-8))

    fig, axes = plt.subplots(4, 1, figsize=(8.8, 10.5), sharex=True)
    axes[0].plot(t, line_phys, label="physics")
    axes[0].plot(t, line_hybrid, label="physics + RBF")
    axes[0].plot(t, line_true, label="hidden true", linestyle="--")
    axes[0].set_ylabel("P_n")
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()

    axes[1].plot(t, log_infidelity(line_phys), label="physics")
    axes[1].plot(t, log_infidelity(line_hybrid), label="physics + RBF")
    axes[1].plot(t, log_infidelity(line_true), label="hidden true", linestyle="--")
    axes[1].set_ylabel("-log10(1 - P_n)")
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()

    axes[2].plot(t, line_hidden_discrepancy, label="hidden true - physics", linestyle="--")
    axes[2].plot(t, line_learned_probability_correction, label="hybrid - physics")
    axes[2].axhline(0.0, color="black", linewidth=1)
    axes[2].set_ylabel("P_n correction")
    axes[2].grid(True, alpha=0.3)
    axes[2].legend()

    axes[3].plot(t, line_support, label="RBF support")
    axes[3].plot(t, line_residual, label="support-gated logit correction")
    axes[3].axhline(0.0, color="black", linewidth=1)
    axes[3].set_xlabel("interpolation: physics-only GRAPE -> best measured pulse")
    axes[3].grid(True, alpha=0.3)
    axes[3].legend()
    plt.show()
else:
    print("Run the loop first.")


## Export current best pulse

`best_controls` is the 80-vector to send to the hardware waveform builder. It is clipped to `[-2, 2]` and follows the same 4 by 20 coefficient convention as the simulator.


In [ ]:
print("best_controls shape:", best_controls.shape)
print("max abs coefficient:", float(jnp.max(jnp.abs(best_controls))))
print("best measured P_n:", best_measured)
print("best true diagnostic P_n:", best_true_diagnostic)
